In [ ]:
import os
import sys
import django

# Allow async operations in this script (use with caution, as it can lead to issues if not handled properly)
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

# Add your Django project path to the Python path
# Option: adjust the path as necessary for your project structure
sys.path.insert(0, os.path.join(os.getenv('HOME'), 'git', 'neoexchange_fresh', 'neoexchange'))

# Set the DJANGO_SETTINGS_MODULE environment variable\n",
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'neox.settings')

# Initialize Django
django.setup()
print('✅ Django successfully configured!')


In [ ]:
import requests
import warnings

import numpy as np
from astropy.table import QTable
from astropy.time import Time

from astrometrics.sources_subs import translate_constraints, fetch_jpl_orbit, random_delay
from astrometrics.time_subs import dttodecimalday
from core.models import Body, SourceMeasurement

In [ ]:
# Change the current working directory to the project root so JS imports work correctly
os.chdir(os.path.join(os.getenv('HOME'), 'git', 'neoexchange_fresh', 'neoexchange'))
from core.views import update_MPC_obs  # noqa: E402


In [ ]:
class JPLSBDBQuery:
    """
    The ``JPLSBDBQuery`` provides an interface to JPL's Small Body Database Query
    via its API interface (https://ssd.jpl.nasa.gov/tools/sbdb_query.html)
    """

    base_url = 'https://ssd-api.jpl.nasa.gov/sbdb_query.api'

    # Maps JPL SBDB orbit class codes to Body OBJECT_SUBTYPES
    SBDB_CLASS_TO_SUBTYPE = {
        # Asteroids
        'IEO': 'N1',  # Atira (Q < 0.983 AU)
        'ATE': 'N2',  # Aten (a < 1.0 AU; Q > 0.983 AU)
        'APO': 'N3',  # Apollo (a > 1.0 AU; q < 1.017 AU)
        'AMO': 'N4',  # Amor (1.017 AU < q < 1.3 AU)
        'IMB': 'MI',  # Inner Main Belt
        'MBA': 'M',   # Main Belt
        'OMB': 'MO',  # Outer Main Belt
        'TJN': 'T4',  # Jupiter Trojan (L4/L5 indistinguishable from class alone)
        'HYA': 'H',   # Hyperbolic asteroid
        'PAA': 'PA',  # Parabolic asteroid
        # Comets
        'JFC': 'JF',  # Jupiter-family (Tisserand-based, T > 2)
        'JFc': 'JF',  # Jupiter-family (period-based, P < 20 y)
        'HTC': 'HT',  # Halley-type
        'HTc': 'HT',  # Halley-type (alternative classification)
        'HYP': 'H',   # Hyperbolic comet
        'PAR': 'PA',  # Parabolic comet
        'DNC': 'DN',  # Dynamically new comet
        'LPC': 'LP',  # Long-period comet
        # No clean mapping for:
        # 'MCA': Mars-crossing (not in OBJECT_SUBTYPES)
        # 'CEN': Centaur (in OBJECT_TYPES as 'E' but absent from OBJECT_SUBTYPES)
        # 'TNO': Trans-Neptunian (ambiguous between 'K' Classical KBO and 'S' SDO)
        # 'AST': Generic asteroid
    }

    def __init__(self, orbit_kind=None, orbit_class=None, orbital_constraints=None):
        """
        orbit_class: str or None (e.g. 'IEO', 'TJN', etc.)
        orbital_constraints: list of constraint strings, e.g. ['q|LT|1.3', 'i|LT|10.5']
        """
        if orbit_class is None and orbital_constraints is None:
            orbital_constraints = ['e>=1.2']
        if orbit_kind not in {'a', 'c', None}:
            raise ValueError(f"orbit_kind must be 'a', 'c', or None, got {orbit_kind!r}")
        self.orbit_kind = orbit_kind
        self.orbit_class = orbit_class
        self.orbital_constraints_raw = orbital_constraints or []
        self.orbital_constraints = translate_constraints(self.orbital_constraints_raw)


    def build_query_url(self):
        """
        Build a query for the JPL SBDB service.
        """
        # Base query fields
        params = {
            'fields': 'pdes,prefix,class,epoch_mjd,e,a,q,i,om,w,tp,H,G,M1,K1,condition_code,data_arc,n_obs_used',
            'full-prec': 'true',
            'sb-xfrag': 'true',
        }

        # Add sb-kind if provided
        if self.orbit_kind:
            params['sb-kind'] = self.orbit_kind

        # Add sb-class if provided
        if self.orbit_class:
            params['sb-class'] = self.orbit_class

        # Add sb-cdata if constraints provided
        if self.orbital_constraints:
            params['sb-cdata'] = self.orbital_constraints

        # Build URL
        query_parts = [f'{key}={str(value)}' for key, value in params.items()]
        url = f'{self.base_url}?' + '&'.join(query_parts)
        self.url = url
        return url

    def run_query(self):
        """
        Execute the query and return results as JSON (if successful).
        """
        url = self.build_query_url()
        resp = requests.get(url)

        if resp.ok:
            return resp.json()
        else:
            print(f'Query failed with status {resp.status_code}')
            return None

    def parse_results(self, results):
        """
        Parse JSON results into an Astropy QTable.
        """
        if not results or 'data' not in results:
            print('No data found in results')
            self.results_table = QTable()
            return self.results_table

        data = results['data']
        columns = results['fields']
        self.results_table = QTable(rows=data, names=columns)
        return self.results_table

    def _get_field(self, result, key, default=None):
        """Safely retrieve a field from a QTable row, returning default for missing or masked values."""
        if key not in self.results_table.colnames:
            return default
        val = result[key]
        if hasattr(val, 'mask') and val.mask:
            return default
        return default if val is None else val

    def create_targets(self) -> list:
        """
        Create NEOx Bodys from JPL SBDB Query. Returns a list of the newly created `Body`s.

        Returns
        -------
        list
            A list of the newly created `Body` objects (or an empty list if the needed `self.results_table`
            is empty.
        """
        from astropy.time import Time

        new_targets = []
        if not getattr(self, 'results_table', None):
            return new_targets
        for result in self.results_table:
            print(f"Processing {result['pdes']}...")
            asteroid = True
            name = result['pdes']
            if result['prefix'] in ['C', 'A', 'P', 'D']:
                if name[-1:] == 'P' and result['prefix'] == 'P':
                    # Numbered periodic comet, don't add prefix
                    pass
                else:
                    name = result['prefix'] + '/' + name
            existing_objects = Body.objects.filter(name=name)
            if existing_objects.count() == 0:
                target = Body()
                if result['prefix'] is None:
                    target.elements_type = 'MPC_MINOR_PLANET'
                else:
                    target.elements_type = 'MPC_COMET'
                    asteroid = False
                target.name = name
                target.source_type = 'A' if asteroid else 'C'
                target.source_subtype_1 = self.SBDB_CLASS_TO_SUBTYPE.get(self._get_field(result, 'class'))

                if self._get_field(result, 'pha') == 'Y' and target.source_subtype_1 in neo_subtypes:
                    target.source_subtype_2 = 'PH'

                # source_subtype_2: flag PHAs where the primary subtype is already a NEO class
                neo_subtypes = {'N1', 'N2', 'N3', 'N4'}
                if self._get_field(result, 'pha') == 'Y' and target.source_subtype_1 in neo_subtypes:
                    target.source_subtype_2 = 'PH'
                target.origin = 'D'
                target.argofperih = result['w']       # argument of perihelion
                target.longascnode = result['om']      # longitude of ascending node
                target.orbinc = result['i']            # inclination
                target.eccentricity = result['e']      # eccentricity
                target.perihdist = result['q']         # perihelion distance
                if asteroid:
                    target.meandist = result['a']      # semi-major axis (asteroids only)
                # Convert epoch of elements from MJD to datetime
                # Wrap in warnings catcher for Erfa 
                with warnings.catch_warnings():
                    warnings.filterwarnings('ignore', category=ErfaWarning)
                    try:
                        target.epochofel = Time(float(result['epoch_mjd']), format='mjd').to_datetime()
                    except (TypeError, ValueError):
                        pass
                    # Convert epoch of perihelion from JD string to datetime.
                    # JPL SBDB's 'tp' field is a plain JD string (e.g.
                    # '2398143.624', confirmed against the live API) - just
                    # subtract the JD->MJD offset directly. (Previously this
                    # sliced off the first 2 characters before converting,
                    # which silently mangled the JD into a wildly wrong date -
                    # e.g. turning 1853 into 2127 - corrupting epochofperih
                    # for every ingested comet.)
                    try:
                        perih_mjd = float(result['tp']) - 2400000.5
                    except (TypeError, ValueError):
                        perih_mjd = None
                    if perih_mjd is not None:
                        target.epochofperih = Time(perih_mjd, format='mjd').to_datetime()
                target.arc_length = result['data_arc']
                target.num_obs = result['n_obs_used']
                # Extract absolute magnitude (H) and slope (G) or M1, K1 for comets
                # Default to G=0.15 for asteroids
                if asteroid:
                    target.abs_mag = result['H']
                    target.slope = result['G'] if result['G'] is not None else 0.15
                else:
                    target.abs_mag = result['M1']
                    target.slope = result['K1']
                target.save()
                new_targets.append(target)
        return new_targets

In [ ]:
from astropy.utils.exceptions import ErfaWarning


comet_classes = 'PAR,HYP,COM'
with warnings.catch_warnings():
    # Catch possible dubious year warnings from erfa
    warnings.filterwarnings('ignore', category=ErfaWarning)
    tp_start = Time('1700-01-01', scale='utc')

    comet_constraints = ['data_arc>7', f'tp>={tp_start.tdb.jd:.2f}']
comet_kind='c'


In [ ]:
jpl = JPLSBDBQuery(orbit_kind=comet_kind, orbit_class=comet_classes, orbital_constraints=comet_constraints)
print(jpl.__dict__)

### Run the query, assert we got the expected API signature so we know the results are going to be what we expect and print count of potential objects

In [ ]:
results = jpl.run_query()

# Doublecheck we got the expected signature before proceeding
assert results['signature'] == {'version': '1.0', 'source': 'NASA/JPL SBDB (Small-Body DataBase) Query API'}

print(f"Retrieved {results['count']} potential comets")
print(results['data'][0:10])

In [ ]:
table = jpl.parse_results(results)
table.sort('tp')


In [ ]:
print(table.colnames)

In [ ]:
target_pdes = '1858 R1'
idx = np.where(table['pdes'] == target_pdes)[0][0]
print(jpl.results_table[idx-2:idx+3][('pdes', 'tp')])

Hand checking some of the early comet results for potential time of perihelion (`tp`) misparsing

In [ ]:
# from django.forms import model_to_dict
for result in jpl.results_table[idx-2:idx+3]:
    print()
    print(f"Processing {result['pdes']}...")
    print(result[('epoch_mjd', 'e', 'a', 'q', 'i', 'om', 'w', 'tp', )])
    asteroid = True
    name = result['pdes']
    if result['prefix'] in ['C', 'A', 'P', 'D']:
        if name[-1:] == 'P' and result['prefix'] == 'P':
            # Numbered periodic comet, don't add prefix
            pass
        else:
            name = result['prefix'] + '/' + name
    existing_target = Body.objects.get(name=name)
    target = Body()
    if result['prefix'] is None:
        target.elements_type = 'MPC_MINOR_PLANET'
    else:
        target.elements_type = 'MPC_COMET'
        asteroid = False
    target.name = name
    target.source_type = 'A' if asteroid else 'C'
    # target.source_subtype_1 = self.SBDB_CLASS_TO_SUBTYPE.get(self._get_field(result, 'class'))

    # if self._get_field(result, 'pha') == 'Y' and target.source_subtype_1 in neo_subtypes:
    #     target.source_subtype_2 = 'PH'

    # # source_subtype_2: flag PHAs where the primary subtype is already a NEO class
    # neo_subtypes = {'N1', 'N2', 'N3', 'N4'}
    # if self._get_field(result, 'pha') == 'Y' and target.source_subtype_1 in neo_subtypes:
    #     target.source_subtype_2 = 'PH'
    target.origin = 'D'
    target.argofperih = result['w']       # argument of perihelion
    target.longascnode = result['om']      # longitude of ascending node
    target.orbinc = result['i']            # inclination
    target.eccentricity = result['e']      # eccentricity
    target.perihdist = result['q']         # perihelion distance
    if asteroid:
        target.meandist = result['a']      # semi-major axis (asteroids only)

    # convert to mjd from jd (preserving precision)
    try:
        perih_mjd  = float(result['tp'][2:]) - 0.5
    except (IndexError, TypeError):
        # Already not a string (or None)
        try:
            perih_mjd = float(result['tp']) - 2400000.5
        except (ValueError, TypeError):
            pass
    target.epochofperih = Time(perih_mjd, format='mjd').to_datetime()
    print(target.epochofperih, Time(float(result['tp']), format='jd').to_datetime(), result['tp'])

In [ ]:
new_targets = jpl.create_targets()
print(f"Created {len(new_targets)} new targets in the database")

In [ ]:
bodies = Body.objects.filter(origin='D')
print(f"Total bodies with origin='D': {bodies.count()}")
for body in bodies.order_by('epochofel'):
    subtype1 = body.source_subtype_1 or '  '
    subtype2 = body.source_subtype_2 or '  '
    print(f"{body.name:>14s}: {body.source_type} {subtype1:2s} {subtype2:2s} {dttodecimalday(body.epochofel)}, {dttodecimalday(body.epochofperih)}", end='')
    measures = update_MPC_obs(body.current_name())
    print(f" #obs={SourceMeasurement.objects.filter(body=body).count()}")
    random_delay(10, 30)


## Determine discovery date/distance and plot

For each comet, find a discovery datetime (preferring the MPC discovery
asterisk on an existing observation, falling back to the MPC Explorer API
and then to the MPC orbit page's "first observation date used" for comets
with no observations at all), compute the heliocentric distance at that
date, and plot discovery distance vs. discovery year.

### One-off correction for already-ingested Bodies

A bug in `JPLSBDBQuery.create_targets()` above (now fixed) mis-parsed JPL's
`tp` (time of perihelion passage) field, corrupting `epochofperih` for every
comet already ingested - e.g. `C/1853 R1` got `2127-08-02` instead of the
correct `1853-10-17`. For comets whose discovery date ends up far from their
(wrong) perihelion epoch, propagating a near-parabolic orbit over that bogus
~270 year gap produces absurd heliocentric distances - this is the ~236 au
"plateau" seen for pre-1860 comets in the plot. Re-fetch the same SBDB query
and fix `epochofperih` on the existing rows before computing any distances.

In [ ]:
jpl_fix = JPLSBDBQuery(orbit_kind=comet_kind, orbit_class=comet_classes, orbital_constraints=comet_constraints)
fix_results = jpl_fix.run_query()
assert fix_results['signature'] == {'version': '1.0', 'source': 'NASA/JPL SBDB (Small-Body DataBase) Query API'}
fix_table = jpl_fix.parse_results(fix_results)
print(f"Re-fetched {len(fix_table)} comets from JPL SBDB for correction pass")

corrected = []
unchanged = []
not_found = []
for result in fix_table:
    name = result['pdes']
    if result['prefix'] in ['C', 'A', 'P', 'D']:
        if name[-1:] == 'P' and result['prefix'] == 'P':
            pass
        else:
            name = result['prefix'] + '/' + name
    try:
        body = Body.objects.get(name=name, origin='D')
    except (Body.DoesNotExist, Body.MultipleObjectsReturned):
        not_found.append(name)
        continue

    try:
        perih_mjd = float(result['tp']) - 2400000.5
    except (TypeError, ValueError):
        continue
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', category=ErfaWarning)
        correct_epochofperih = Time(perih_mjd, format='mjd').to_datetime()

    if body.epochofperih is None or abs((body.epochofperih - correct_epochofperih).total_seconds()) > 86400:
        body.epochofperih = correct_epochofperih
        body.save()
        corrected.append(name)
    else:
        unchanged.append(name)

print(f"Corrected epochofperih for {len(corrected)} bodies")
print(f"Already correct: {len(unchanged)} bodies")
print(f"Not matched to an existing origin='D' Body: {len(not_found)}")
if not_found:
    print(not_found[:20], '...' if len(not_found) > 20 else '')

In [ ]:
other_bodies = Body.objects.filter(name__in=not_found)
print(f"# other bodies= {other_bodies.count()} # not found = {len(not_found)}")

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
# core.views (imported below, and elsewhere in this notebook) pulls in
# photometrics.SA_scatter, which calls matplotlib.use('Agg') for the
# non-interactive web app - switch back to an inline backend here so plt.show()
# actually displays figures in this notebook.

from astropy.time import Time
from datetime import datetime

from astrometrics.sources_subs import fetch_mpcdb_page, parse_mpcorbit
from core.views import update_MPC_obs_from_api

In [ ]:
def get_discovery_measurement(body):
    """Find the SourceMeasurement to treat as the discovery observation for
    `body`: the one flagged with the MPC discovery asterisk if there is one,
    else the earliest measurement by date as a proxy. Returns None if `body`
    has no SourceMeasurements at all.
    """
    measurements = SourceMeasurement.objects.filter(body=body).select_related('frame')
    discovery = measurements.filter(flags__startswith='*').order_by('frame__midpoint').first()
    if discovery:
        return discovery
    return measurements.order_by('frame__midpoint').first()

In [ ]:
def get_discovery_date_and_distance(body, skip_api_fetch=False, dry_run=False):
    """3-tier fallback to find a discovery datetime for `body`, then compute
    its heliocentric distance at that date.

    Tier 1: an existing SourceMeasurement (no network call).
    Tier 2: fetch observations via the MPC Explorer API (update_MPC_obs_from_api),
        then re-check for a SourceMeasurement. Skipped entirely if
        `skip_api_fetch` is True (e.g. once this has already been run for the
        full dataset and we don't want to hammer MPC's API again).
    Tier 3: scrape the MPC orbital-elements page for "first observation date
        used" (covers comets with no observations available via either of
        the above, e.g. very old/obscure ones).

    If `dry_run` is True, no network calls are made at all (tiers 2 and 3 are
    not attempted, regardless of `skip_api_fetch`): bodies resolved by tier 1
    still get their real result, but anything that would otherwise need a
    network call gets a placeholder dict with source_tier='would_call_mpc'
    and discovery_dt/decimal_year/helio_dist left as None. This lets you
    count/list how many MPC calls a real run would make without making them.

    Returns a dict with body/name/discovery_dt/decimal_year/helio_dist/
    source_tier, or None if no usable date or distance could be determined
    (not applicable in dry_run mode, which never returns None).
    """
    name = body.current_name()
    discovery_dt = None
    tier = None

    measurement = get_discovery_measurement(body)
    if measurement:
        discovery_dt = measurement.frame.midpoint
        tier = 'obs'

    if discovery_dt is None and dry_run:
        return {
            'body': body,
            'name': name,
            'discovery_dt': None,
            'decimal_year': None,
            'helio_dist': None,
            'source_tier': 'would_call_mpc',
        }

    if discovery_dt is None and not skip_api_fetch:
        if update_MPC_obs_from_api(name):
            random_delay(10, 30)
            measurement = get_discovery_measurement(body)
            if measurement:
                discovery_dt = measurement.frame.midpoint
                tier = 'api'
        else:
            random_delay(10, 30)

    if discovery_dt is None:
        page = fetch_mpcdb_page(name)
        random_delay(10, 30)
        if page is not None:
            elements = parse_mpcorbit(page)
            first_obs = elements.get('first observation date used')
            if first_obs:
                try:
                    discovery_dt = datetime.strptime(first_obs.replace('.0', ''), '%Y-%m-%d')
                    tier = 'first_obs_used'
                except ValueError:
                    pass

    if discovery_dt is None:
        return None

    # Old/obscure comets (e.g. C/1702 H1) have discovery dates far enough in
    # the past to trip erfa's "dubious year" check, both in the ephemeris
    # computation and the Time(...).decimalyear conversion below. Silence those
    # here, matching the ErfaWarning handling in the ingest/correction cells.
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', category=ErfaWarning)
        distances = body.compute_distances(discovery_dt)
        if not distances:
            return None
        geocentric_dist, helio_dist = distances
        decimal_year = Time(discovery_dt).decimalyear

    return {
        'body': body,
        'name': name,
        'discovery_dt': discovery_dt,
        'decimal_year': decimal_year,
        'helio_dist': helio_dist,
        'source_tier': tier,
    }


In [ ]:
import csv
from datetime import datetime

RESULTS_CSV_PATH = os.path.join('notebooks', 'discovery_dist_results.csv')

def save_results(results, skipped, path=RESULTS_CSV_PATH):
    """Save `results` (list of dicts from get_discovery_date_and_distance)
    and `skipped` (list of body names with no result) to a CSV checkpoint,
    so a long batched run doesn't have to be repeated just to replot.
    """
    with open(path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['name', 'body_id', 'discovery_dt', 'decimal_year', 'helio_dist', 'source_tier'])
        for r in results:
            writer.writerow([
                r['name'],
                r['body'].id,
                r['discovery_dt'].isoformat() if r['discovery_dt'] else '',
                r['decimal_year'] if r['decimal_year'] is not None else '',
                r['helio_dist'] if r['helio_dist'] is not None else '',
                r['source_tier'],
            ])
        for name in skipped:
            writer.writerow([name, '', '', '', '', 'skipped'])
    print(f"Saved {len(results)} results and {len(skipped)} skipped to {path}")


def load_results(path=RESULTS_CSV_PATH):
    """Reload a checkpoint saved by save_results(). Returns (results, skipped)
    in the same shape as a live run, except 'body' is replaced by 'body_id'
    (re-fetch via Body.objects.get(pk=body_id) if you ever need the live
    object - the plotting cell only needs name/decimal_year/helio_dist, so
    it works unchanged on a reloaded `results`).
    """
    results = []
    skipped = []
    with open(path, newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row['source_tier'] == 'skipped':
                skipped.append(row['name'])
                continue
            results.append({
                'name': row['name'],
                'body_id': int(row['body_id']),
                'discovery_dt': datetime.fromisoformat(row['discovery_dt']) if row['discovery_dt'] else None,
                'decimal_year': float(row['decimal_year']) if row['decimal_year'] else None,
                'helio_dist': float(row['helio_dist']) if row['helio_dist'] else None,
                'source_tier': row['source_tier'],
            })
    print(f"Loaded {len(results)} results and {len(skipped)} skipped from {path}")
    return results, skipped

In [ ]:
from django.db.models import Count

from core.models import Designations


def sbdb_target_name(result):
    """Build the NEOx Body.name for a JPL SBDB result row, applying the same
    comet prefix logic used by JPLSBDBQuery.create_targets() when the targets
    were ingested."""
    name = result['pdes']
    if result['prefix'] in ['C', 'A', 'P', 'D']:
        if name[-1:] == 'P' and result['prefix'] == 'P':
            # Numbered periodic comet, don't add prefix
            pass
        else:
            name = result['prefix'] + '/' + name
    return name


def resolve_body(name):
    """Pick the best Body for `name` when duplicates exist (Body.name has no
    uniqueness constraint): the one with the most local SourceMeasurements,
    tie-broken by most recently updated, then newest ingest.

    Ranking by SourceMeasurement count both maximizes the chance of finding a
    real discovery observation without an MPC call (tier 1 of
    get_discovery_date_and_distance) and selects the canonical, actively-
    followed body - whose epochofperih came cleanly from MPC rather than the
    create_targets() tp-parsing bug that the correction cell above patched on
    the origin='D' rows.

    If no Body has `name` as its primary name, fall back to a Designations
    lookup: the SBDB pdes may be a non-primary designation of a Body that has
    since been renamed (e.g. C/2025 N1 -> 3I/ATLAS).

    Returns (body, n_matches); body is None if `name` matches no Body at all.
    """
    def ranked(qs):
        return (qs.annotate(n_sm=Count('sourcemeasurement'))
                  .order_by('-n_sm', '-update_time', '-ingest'))

    qs = ranked(Body.objects.filter(name=name))
    if not qs.exists():
        ids = Designations.objects.filter(value=name).values_list('body_id', flat=True)
        qs = ranked(Body.objects.filter(id__in=ids))
    return qs.first(), qs.count()


In [ ]:
from collections import Counter

from django.db import close_old_connections

# Set to True to skip the MPC Explorer API fetch (tier 2) entirely, e.g. if
# it's already been run once for this dataset and you want to avoid further
# load on MPC's API. Ignored if dry_run is True (no network calls happen).
skip_api_fetch = False

# Set to True to make NO network calls at all: just count/list how many
# bodies would need at least one MPC call (tier 2 and/or 3) in a real run,
# without actually making any of those calls.
dry_run = False

# Process comets in batches, with a longer pause between batches (on top of
# the per-call random_delay(10,30) already inside tiers 2/3), to avoid
# hammering MPC continuously over the whole run.
batch_size = 50
pause_between_batches = (120, 180)  # seconds, passed to random_delay()

# Drive the target list from the JPL SBDB query results, not from
# Body.objects.filter(origin='D', ...). Comets already in the DB under another
# origin (e.g. C/2014 UN271) were skipped by create_targets() and so never got
# an origin='D' row, but still belong in the plot. resolve_body() maps each
# SBDB name to the best matching Body, handling duplicate names.
sbdb_names = [sbdb_target_name(result) for result in jpl.results_table]

comet_bodies = []
not_in_db = []
ambiguous = []
for name in sbdb_names:
    body, n_matches = resolve_body(name)
    if body is None:
        not_in_db.append(name)
        continue
    if n_matches > 1:
        ambiguous.append((name, n_matches))
    comet_bodies.append(body)

total = len(comet_bodies)
print(f"SBDB query returned {len(sbdb_names)} comets")
print(f"Matched {total} to a Body; {len(not_in_db)} not in DB; "
      f"{len(ambiguous)} names had duplicates resolved")
if not_in_db:
    print("Not in DB:", not_in_db[:20], '...' if len(not_in_db) > 20 else '')
if ambiguous:
    print("Ambiguous (name, #rows):", ambiguous[:20], '...' if len(ambiguous) > 20 else '')

results = []
skipped = []
for batch_start in range(0, total, batch_size):
    batch = comet_bodies[batch_start:batch_start + batch_size]
    # Drop any DB connection that went idle during the previous batch pause /
    # MPC waits and was closed server-side (pgbouncer); Django reopens lazily.
    close_old_connections()
    batch_num = batch_start // batch_size + 1
    num_batches = (total + batch_size - 1) // batch_size
    print(f"Batch {batch_num}/{num_batches} ({batch_start}-{batch_start + len(batch) - 1} of {total})...")
    for body in batch:
        info = get_discovery_date_and_distance(body, skip_api_fetch=skip_api_fetch, dry_run=dry_run)
        if info:
            results.append(info)
        else:
            skipped.append(body.current_name())
    if batch_start + batch_size < total:
        print(f"Batch {batch_num} done. Pausing before next batch...")
        delay = random_delay(*pause_between_batches)
        print(f"Paused {delay}s.")

tier_counts = Counter(r['source_tier'] for r in results)
print(f"Resolved {len(results)} of {total} bodies")
print(f"By tier: {dict(tier_counts)}")
print(f"Skipped/failed: {len(skipped)}")
if skipped:
    print(skipped[:20], '...' if len(skipped) > 20 else '')

if dry_run:
    needs_mpc = [r['name'] for r in results if r['source_tier'] == 'would_call_mpc']
    print(f"\nDry run: {len(needs_mpc)} bodies would need at least one MPC call (tier 2 and/or 3).")
    print(needs_mpc[:20], '...' if len(needs_mpc) > 20 else '')
else:
    save_results(results, skipped)


In [ ]:
# Resume the batch loop above after a dropped DB connection WITHOUT re-running
# cell f15dcd18 (which would reset results/skipped and re-do everything).
# `results` and `skipped` are still in memory from the partial run.
from django.db import connection

# The Postgres connection went idle and was closed server-side (pgbouncer)
# during the long random_delay sleeps. Drop the dead handle; Django reopens
# lazily on the next ORM call.
connection.close()

# Each processed body appended exactly one entry (to results OR skipped) in
# order, so comet_bodies[processed:] is what's left. The body that errored was
# never recorded, so it gets retried.
processed = len(results) + len(skipped)
remaining = comet_bodies[processed:]
print(f"Already processed {processed} of {len(comet_bodies)}; resuming on {len(remaining)}")

for batch_start in range(0, len(remaining), batch_size):
    batch = remaining[batch_start:batch_start + batch_size]
    # Proactively drop any connection that went idle during the last pause /
    # MPC waits, so we reconnect fresh rather than hitting a dead socket.
    connection.close()
    abs_start = processed + batch_start
    print(f"Resuming {abs_start}-{abs_start + len(batch) - 1} of {len(comet_bodies)}...")
    for body in batch:
        info = get_discovery_date_and_distance(body, skip_api_fetch=skip_api_fetch, dry_run=dry_run)
        if info:
            results.append(info)
        else:
            skipped.append(body.current_name())
    if batch_start + batch_size < len(remaining):
        delay = random_delay(*pause_between_batches)
        print(f"Paused {delay}s.")

tier_counts = Counter(r['source_tier'] for r in results)
print(f"Resolved {len(results)} of {len(comet_bodies)} bodies")
print(f"By tier: {dict(tier_counts)}")
print(f"Skipped/failed: {len(skipped)}")
if skipped:
    print(skipped[:20], '...' if len(skipped) > 20 else '')
if not dry_run:
    save_results(results, skipped)


In [ ]:
# Optional: reload a previously saved checkpoint instead of re-running the
# batch loop above (e.g. in a fresh kernel session). Uncomment to use:
# results, skipped = load_results()

In [ ]:
from matplotlib.ticker import MultipleLocator

# Font sizes/family chosen to match the original discovery_dist.pdf (LOOK Year
# 1 report figure), which was typeset with LaTeX in Computer Modern Roman
# (CMR12/CMR8/CMR5): ~12pt axis labels + tick labels, ~11pt comet annotations.
LABEL_SIZE = 12
TICK_SIZE = 12
ANNOT_SIZE = 11

# Render text through LaTeX in Computer Modern to match the original figure.
# Scoped via rc_context so usetex doesn't leak into other cells.
PLOT_RC = {
    'text.usetex': True,
    'font.family': 'serif',
}

# Label placement: (dx, dy) in data units added to the point.
#  - 3I sits low among the dense post-2000 cloud, so lift its label above-left.
#  - C/2014 UN271 is the topmost point (29 au); the default +dy clips above the
#    axes, so drop its label below-left to keep it inside the bounds.
DEFAULT_OFFSET = (-30, 1.5)
ANNOT_OFFSETS = {
    '3I': (-30, 9),
    'C/2014 UN271': (-38.85, -0.75),
}

years = [r['decimal_year'] for r in results]
dists = [r['helio_dist'] for r in results]
top_n = sorted(results, key=lambda r: r['helio_dist'], reverse=True)[:3]
# Always annotate 3I (interstellar) too, even if it's not in the top 3.
annotated = list(top_n)
three_i = next((r for r in results if r['name'] == '3I'), None)
if three_i and three_i not in annotated:
    annotated.append(three_i)


def make_plot(filename, xstart=None, xend=None, xmajor=None, extra_ticks=None):
    """Scatter of discovery heliocentric distance vs discovery date, saved to
    `filename`. y-axis starts at 0. Optionally clip the x-axis to [xstart, xend],
    place major ticks every `xmajor` years, and add any `extra_ticks`."""
    with plt.rc_context(PLOT_RC):
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.scatter(years, dists, marker='x', alpha=0.5)

        ax.set_xlabel('discovery date', fontsize=LABEL_SIZE)
        ax.set_ylabel('discovery heliocentric distance [au]', fontsize=LABEL_SIZE)
        ax.tick_params(labelsize=TICK_SIZE)
        ax.set_ylim(bottom=0)
        if xstart is not None:
            ax.set_xlim(left=xstart)
        if xend is not None:
            ax.set_xlim(right=xend)
        if xmajor is not None:
            if extra_ticks:
                lo = xstart if xstart is not None else min(years)
                hi = xend if xend is not None else max(years)
                ticks = sorted(set(np.arange(lo, hi + 1, xmajor)) | set(extra_ticks))
                ax.set_xticks(ticks)
            else:
                ax.xaxis.set_major_locator(MultipleLocator(xmajor))

        for r in annotated:
            dx, dy = ANNOT_OFFSETS.get(r['name'], DEFAULT_OFFSET)
            ax.annotate(
                r['name'],
                xy=(r['decimal_year'], r['helio_dist']),
                xytext=(r['decimal_year'] + dx, r['helio_dist'] + dy),
                arrowprops=dict(arrowstyle='->'),
                fontsize=ANNOT_SIZE,
            )

        plt.tight_layout()
        plt.savefig(os.path.join('notebooks', filename))
        plt.show()


# Full range (x-axis capped at 2030)
make_plot('discovery_dist_v2.pdf', xend=2030)
# Clipped to 1800-2030, 40-year major ticks plus an extra tick at 2020
make_plot('discovery_dist_v2_from1800.pdf', xstart=1800, xend=2030, xmajor=40, extra_ticks=[2020])
